# 实践 13.2  梯度下降与不平衡分类评价

《统计遗传与智能育种学》第十三章 · 电子资源 `13b_ai_foundations.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：13b_ai_foundations.ipynb。实践固定随机种子，完整展示数据划分、训练集标准化、梯度更新、验证与测试评价，并通过类别不平衡示例说明较高准确率仍可能伴随阳性类别召回不足。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果：SEED=1302。梯度下降的均方误差（MSE）由8.1531降至0.5025，估计b=1.851、w=2.394。分类train/validation/test=960/320/320，validation AP选中C=10；test阳性率=0.066，多数类accuracy基线=0.934，模型accuracy=0.941、ROC AUC=0.686、AP=0.220、precision=1.000、recall=0.095。因此，94.1%的accuracy不能单独解释为模型对阳性位点具有良好的识别性能。

## 输入、完整代码、实际输出与解释

输入与可复现条件：300个梯度下降回归样本；1600个不平衡分类样本、20个特征、约6%阳性；SEED=1302；无外部文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED = 1302
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1302


In [2]:
# 一元线性回归的梯度下降
x=rng.normal(size=300)
y=1.8+2.4*x+rng.normal(0,0.7,300)
w=0.0
b=0.0
lr=0.05
loss=[]
for epoch in range(180):
    pred=b+w*x
    err=pred-y
    loss.append(float(np.mean(err**2)))
    b-=lr*2*np.mean(err)
    w-=lr*2*np.mean(err*x)
print('start loss',round(loss[0],4),'final loss',round(loss[-1],4),
      'b',round(b,3),'w',round(w,3))


start loss 8.1531 final loss 0.5025 b 1.851 w 2.394


In [3]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score,roc_auc_score,average_precision_score,
                             precision_score,recall_score)

X,y=make_classification(n_samples=1600,n_features=20,n_informative=6,
    weights=[0.94,0.06],class_sep=0.8,random_state=SEED)
idx=np.arange(len(y))
dev,test=train_test_split(idx,test_size=0.20,stratify=y,random_state=SEED)
train,val=train_test_split(dev,test_size=0.25,stratify=y[dev],random_state=SEED+1)
print('split sizes train/validation/test',len(train),len(val),len(test))

candidates=[0.1,1,10]
val_ap=[]
for C in candidates:
    clf=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=1000))
    clf.fit(X[train],y[train])
    p=clf.predict_proba(X[val])[:,1]
    ap=float(average_precision_score(y[val],p))
    val_ap.append(ap)
    print('C',C,'validation average precision',round(ap,3))
best_C=candidates[int(np.argmax(val_ap))]

clf=make_pipeline(StandardScaler(),LogisticRegression(C=best_C,max_iter=1000))
clf.fit(X[dev],y[dev])
prob=clf.predict_proba(X[test])[:,1]
pred=(prob>=0.5).astype(int)
prevalence=float(y[test].mean())
print('selected C',best_C,'test positive prevalence',round(prevalence,3))
print('majority-class accuracy baseline',round(1-prevalence,3))
print('test accuracy',round(float(accuracy_score(y[test],pred)),3),
      'ROC AUC',round(float(roc_auc_score(y[test],prob)),3),
      'average precision',round(float(average_precision_score(y[test],prob)),3))
print('test precision',round(float(precision_score(y[test],pred,zero_division=0)),3),
      'recall',round(float(recall_score(y[test],pred)),3))


split sizes train/validation/test 960 320 320
C 0.1 validation average precision 0.151
C 1 validation average precision 0.192
C 10 validation average precision 0.194
selected C 10 test positive prevalence 0.066
majority-class accuracy baseline 0.934
test accuracy 0.941 ROC AUC 0.686 average precision 0.22
test precision 1.0 recall 0.095


结果解释：本例准确率比全判阴性的基线高约0.6个百分点，而召回率只有0.095。ROC AUC与AP说明模型存在一定排序信息，但当前0.5阈值下阳性召回率很低。实际筛选应根据候选量、验证成本和漏检代价在验证数据中确定阈值。

可修改练习：把阳性比例从0.06改为0.02，完整重新执行验证集选参与测试集评价；再把分类阈值从0.5改为0.2，在验证集比较准确率、精确率和召回率。